# Classificazione con alberi decisionali

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

---

## Caricamento e ispezione dei dati

Questo dataset contiene le descrizioni di campioni ipotetici corrispondenti a 23 specie di funghi lamellati delle famiglie Agaricus e Lepiota (pp. 500-525). Ogni specie è identificata come sicuramente commestibile, sicuramente velenosa, oppure di commestibilità sconosciuta e non raccomandata. Quest'ultima classe è stata unita a quella dei funghi velenosi. La Guida afferma chiaramente che non esiste una regola semplice per determinare la commestibilità di un fungo; nessuna regola del tipo "tre foglioline, lascia stare" per la quercia velenosa e l'edera.

In [ ]:
path_csv = "https://raw.githubusercontent.com/LorenzoTonet/Intro_to_ML_26-27/refs/heads/main/data/Mushroom.csv"
df = pd.read_csv(path_csv)
df.head()

In [ ]:
class_counts = df['CLASS'].value_counts()
class_counts.plot(kind='bar')
plt.title("Distribuzione delle classi"), 
plt.xlabel("Classe"), 
plt.ylabel("Numero di esempi")

In [ ]:
df.shape

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Divido il dataset in train e test (70% / 30%)
X_train, X_test, y_train, y_test = train_test_split(df.drop("CLASS", axis=1), df["CLASS"], test_size=0.3, random_state=67)

## Alberi decisionali

**Uso degli alberi decisionali per la classificazione con ID3 (Iterative Dichotomiser 3)**

Ecco una panoramica:

**Promemoria:**

- **Costruzione dell'albero:**
  - Costruisco un albero decisionale a partire da un dataset.
  - Ogni nodo dell'albero è come un punto di controllo: mi aiuta a prendere una decisione (nodo interno) oppure rappresenta un esito (foglia).

- **Algoritmo ID3:**
  - ID3 è un approccio top-down (parto dalla radice) e un algoritmo greedy (considero solo il passo corrente quando scelgo la feature migliore) per costruire alberi decisionali.

- **Selezione delle feature:**
  - A ogni passo, le feature sono raggruppate in base al **guadagno informativo**.
  - Il guadagno informativo mi aiuta a decidere quale feature è la più utile per prendere decisioni.
  - **Formula dell'entropia:** $H(S) = \sum_{i=1}^D -p_i \log p_i$, dove $p_i$ è la proporzione di ogni categoria.
  - **Formula del guadagno informativo:** $IG(S, j) = H(S) - \sum_j \frac{|S_j|}{|S|}H(S_j)$

- **Selezione del nodo:**
  - Un nodo diventa una foglia se tutti i dati in quel nodo appartengono alla stessa classe.

- **Costruzione dell'albero:**
  - Ripeto questo processo finché l'albero non ha solo foglie oppure finché non ho usato tutte le feature disponibili.

Parto dall'alto e prendo decisioni a ogni passo in base alle informazioni più utili. Continuo finché tutte le decisioni (foglie) sono chiare oppure finché non ho esaurito tutte le informazioni disponibili.

### Suggerimenti
* `.unique()` restituisce un iteratore con tutti i valori della serie (lo split di ID3 si basa su un ramo per ogni categoria)
* `.nunique()` restituisce il numero di valori unici (utile per verificare se un sottoinsieme è "puro")

In [ ]:
class ID3DecisionTree:
    """
    L'albero è un dizionario annidato: {feature: {valore: sottoalbero_o_etichetta}}
    """

    def __init__(self):
        self.tree = None

    # ---------- 1. Entropia e guadagno informativo ----------
    def entropy(self, y):
        """H(S) = - somma_c p_c * log2(p_c)"""
        h = 0
        # TODO: calcolo l'entropia delle etichette y

        # 1. Calcolo la proporzione di ogni classe
        probs = y.value_counts(normalize=True)

        # 2. Sommo -p * log2(p) su tutte le classi
        for p in probs:
            if p > 0:
                h -= p * np.log2(p)
        return h

    def information_gain(self, X, y, feature):
        """IG = H(S) - somma_v |S_v|/|S| * H(S_v)"""
        weighted_entropy = 0
        n = len(y)

        # 1. Per ogni valore della feature, calcolo l'entropia del sottoinsieme
        for value in X[feature].unique():
            mask = X[feature] == value
            y_sub = y[mask]

            # 2. Peso l'entropia per la proporzione del sottoinsieme
            weighted_entropy += (len(y_sub) / n) * self.entropy(y_sub)
        return self.entropy(y) - weighted_entropy

    def best_feature(self, X, y):
        """Feature con il guadagno informativo più alto."""
        best_feature = None
        best_gain = -1

        # 1. Calcolo il guadagno informativo di ogni feature
        for feature in X.columns:
            gain = self.information_gain(X, y, feature)

            # 2. Tengo la feature con guadagno massimo
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
        return best_feature

    # ---------- 2. Addestramento ----------
    def build_tree(self, X, y):
        # Stop 1: tutti i punti hanno la stessa etichetta -> foglia
        if y.nunique() == 1:
            return y.iloc[0]

        # Stop 2: nessuna feature rimasta -> foglia con l'etichetta più frequente
        if len(X.columns) == 0:
            return y.mode()[0]

        # Altrimenti: divido sulla feature migliore, un ramo per ogni suo valore
        feature = self.best_feature(X, y)
        branches = {}
        for value in X[feature].unique():
            mask = X[feature] == value

            # 1. Rimuovo la feature già usata per lo split dai dati del sottoalbero
            X_sub = X[mask].drop(columns=feature)
            y_sub = y[mask]
            
            # 2. Costruisco ricorsivamente il sottoalbero
            branches[value] = self.build_tree(X_sub, y_sub)
        return {feature: branches}

    def fit(self, X, y):
        """Addestro l'albero memorizzando la classe di default per i valori mai visti."""
        self.default_class = y.mode()[0]   # usata se un valore non è mai comparso in training
        self.tree = self.build_tree(X, y)
        return self

    # ---------- 3. Predizione ----------
    def predict_one(self, x, node):
        """Predico la classe di un singolo punto scendendo l'albero."""
        if not isinstance(node, dict):          # una foglia: è la risposta
            return node
        feature = list(node.keys())[0]          # feature testata in questo nodo
        branches = node[feature]
        value = x[feature]                      # valore di quella feature per questo punto
        if value in branches:
            return self.predict_one(x, branches[value])   # seguo il ramo
        return self.default_class

    def predict(self, X):
        """Predico la classe per ogni punto del dataset X."""
        predictions = []
        for i in range(len(X)):
            predictions.append(self.predict_one(X.iloc[i], self.tree))
        return predictions

    def score(self, X, y):
        """Calcolo l'accuratezza: proporzione di predizioni corrette."""
        predictions = self.predict(X)
        correct = 0
        for pred, true in zip(predictions, y):
            if pred == true:
                correct += 1
        return correct / len(y)

    # ---------- 4. Stampa ----------
    def print_tree(self, node=None, depth=0):
        """Stampo l'albero in forma testuale indentata."""
        if node is None:
            node = self.tree
        feature = list(node.keys())[0]
        for value, child in node[feature].items():
            print("    " * depth + f"se {feature} = {value}:")
            if isinstance(child, dict):
                self.print_tree(child, depth + 1)
            else:
                print("    " * (depth + 1) + f"-> {child}")

In [ ]:
tree = ID3DecisionTree()
tree.fit(X_train, y_train)

print("Albero decisionale:")
tree.print_tree()

In [ ]:
# 1. Predico sul test set e stampo l'accuratezza
y_pred = tree.predict(X_test)
print(f"Accuratezza sul test set: {tree.score(X_test, y_test):.2f}")

In [ ]:
# TODO: calcolo la matrice di confusione e la stampo

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Calcolo la matrice di confusione confrontando etichette vere e predette
cm = confusion_matrix(y_test, y_pred)

# 2. La stampo come matrice numerica
print("Matrice di confusione:")
print(cm)

# 3. La mostro anche graficamente
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y_test.unique()))
disp.plot(cmap="Blues")
plt.title("Matrice di confusione")
plt.show()

---

### Come sempre...

Dai un'occhiata alla [documentazione ufficiale](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) per vedere tutti i diversi parametri.

### Indice di Gini negli alberi decisionali

Nei problemi di classificazione, l'indice di Gini misura la probabilità di classificare in modo errato un'osservazione scelta a caso, se la sua etichetta di classe venisse assegnata a caso secondo la distribuzione delle classi all'interno del nodo.

La formula è:

$$
Gini = 1 - \sum_{i=1}^{K} p_i^2
$$

dove:
- $K$ è il numero di classi nel nodo;
- $p_i$ è la proporzione di osservazioni che appartengono alla classe $i$.

### Interpretazione

- **Gini = 0**: il nodo è puro, cioè tutte le osservazioni appartengono alla stessa classe.
- **Valori di Gini più alti**: il nodo contiene osservazioni appartenenti a classi diverse ed è quindi più impuro.
- **Impurezza massima**: si verifica quando tutte le classi sono ugualmente rappresentate. Con $K$ classi, il valore massimo è $1 - \frac{1}{K}$.

In [ ]:
# 1. Albero decisionale con scikit-learn
from sklearn.tree import DecisionTreeClassifier

# 2. Codifico le variabili categoriche con get_dummies
X = pd.get_dummies(df.drop(columns="CLASS"))
y = df["CLASS"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=67)

# 3. Addestro l'albero (criterio: entropia, profondità massima 100)
sk_tree = DecisionTreeClassifier(criterion='entropy', max_depth=100, min_samples_split=2, min_samples_leaf=1,)
sk_tree.fit(X_train, y_train)
print(sk_tree.score(X_test, y_test))

---